In [1]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd

# =========================================================
# 0) 配置区
# =========================================================
FOLD_DIR = "../AD/best_xgb_5fold_saved"
N_FOLDS = 5

# 老港特征文件：第1列 SMILES，第2列开始全特征
LAOGANG_FEATURE_FILE = "./laogang_ALL_features.xlsx"

# 老港模型预测二值文件：第1列 SMILES，后面为24个预测标签
LAOGANG_PRED_BIN = "./laogang_to_predict__pred/laogang_ALL_features_pred_binary_thr0.5.xlsx"

SHEET_NAME = 0
OUT_PATH = "./laogang_AD_malodor_hit_summary.xlsx"

# 你的最优 AD 参数
A_WEIGHT = 100.0
EPSILON = 1e-6
K_RATIO = 0.05
RHO_TH = 8.445e-08
IA_TH = 0.4438

LABEL_DIFF_MODE = "jaccard"
K_SWD_CAP = 300

# 24 个气味标签
MALODOR_LABELS_24 = [
    "alcoholic", "aldehydic", "almond", "aromatic", "burnt", "cabbage",
    "cheesy", "cherry", "chocolate", "ethereal", "fishy", "fruity",
    "garlic", "grassy", "green", "ketonic", "musty", "pungent",
    "sharp", "solvent", "sour", "sulfurous", "sweaty", "sweet"
]


# =========================================================
# 1) 读取 fold npz，拼接训练参考集
# =========================================================
def load_fold_npz(fold_id: int):
    path = os.path.join(FOLD_DIR, f"fold_{fold_id:02d}.npz")
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到 fold 文件：{path}")

    z = np.load(path, allow_pickle=True)
    return z["X_tr"], z["y_tr"], z["X_va"], z["y_va"], z["y_prob_va"]


Xtr_list, ytr_list = [], []

for f in range(1, N_FOLDS + 1):
    X_tr, y_tr, _, _, _ = load_fold_npz(f)
    Xtr_list.append(X_tr)
    ytr_list.append(y_tr)

X_tr_all = np.vstack(Xtr_list).astype(np.float32)
y_tr_all = np.vstack(ytr_list).astype(np.int8)

print("[INFO] AD reference train X:", X_tr_all.shape)
print("[INFO] AD reference train y:", y_tr_all.shape)

if y_tr_all.shape[1] != 24:
    raise ValueError(
        f"fold 中的 y_tr 不是 24 列，当前为 {y_tr_all.shape[1]} 列。"
        "请确认 AD 缓存是否来自当前只预测24个气味标签的模型。"
    )


# =========================================================
# 2) AD 计算函数
# =========================================================
def l2_normalize(X, eps=1e-12):
    X = X.astype(np.float32, copy=False)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (n + eps)


def cosine_sim_query(Xq_n, Xtr_n):
    return np.clip(Xq_n @ Xtr_n.T, 0.0, 1.0).astype(np.float32)


def compute_weights(sim_vals, a, epsilon):
    s = np.clip(sim_vals, epsilon, 1.0)
    return np.exp(-a * (1.0 - s) / s).astype(np.float32)


def pairwise_label_diff_scalar(y_t, y_nei, mode="jaccard", eps=1e-12):
    y_t = y_t.astype(np.int8, copy=False)
    y_nei = y_nei.astype(np.int8, copy=False)

    if mode == "hamming":
        return np.mean(np.abs(y_nei - y_t[None, :]), axis=1).astype(np.float32)

    if mode == "jaccard":
        inter = np.sum((y_nei == 1) & (y_t[None, :] == 1), axis=1).astype(np.float32)
        union = np.sum((y_nei == 1) | (y_t[None, :] == 1), axis=1).astype(np.float32)
        j = inter / (union + eps)
        return (1.0 - j).astype(np.float32)

    raise ValueError("mode must be 'jaccard' or 'hamming'")


def compute_SWD_topk(Xtr, ytr, k_swd_eff, a, eps):
    Xn = l2_normalize(Xtr)
    sim = cosine_sim_query(Xn, Xn)

    N = ytr.shape[0]
    SWD = np.zeros(N, dtype=np.float32)
    k_eff = max(1, min(k_swd_eff, N - 1))

    for t in range(N):
        sims_t = sim[t]

        idxs = np.argpartition(sims_t, -(k_eff + 1))[-(k_eff + 1):]
        idxs = idxs[idxs != t]

        if idxs.size == 0:
            continue

        sv = sims_t[idxs]
        w = compute_weights(sv, a, eps)
        diffs = pairwise_label_diff_scalar(ytr[t], ytr[idxs], mode=LABEL_DIFF_MODE)

        SWD[t] = float((w * sv * diffs).sum() / (w.sum() + eps))

    return SWD, Xn


def compute_rho_IA_given_SWD(Xtr_n, SWD, Xq, k_eff, a, eps):
    Xq_n = l2_normalize(Xq)
    sim_qt = cosine_sim_query(Xq_n, Xtr_n)

    M = Xq.shape[0]
    rho = np.zeros(M, dtype=np.float32)
    IA = np.zeros(M, dtype=np.float32)

    for i in range(M):
        sims = sim_qt[i]

        idxk = np.argpartition(sims, -k_eff)[-k_eff:]
        sk = sims[idxk]
        w = compute_weights(sk, a, eps)

        rho[i] = float(w.mean())
        IA[i] = float((w * SWD[idxk]).sum() / (w.sum() + eps))

    return rho, IA


# =========================================================
# 3) 读取老港特征矩阵 X_lg
# =========================================================
df_feat = pd.read_excel(LAOGANG_FEATURE_FILE, sheet_name=SHEET_NAME)

if df_feat.shape[1] < 2:
    raise ValueError("老港特征文件列数不足：至少需要第1列 SMILES + 第2列开始的特征列。")

smiles_feat = df_feat.iloc[:, 0].astype(str)
Xdf = df_feat.iloc[:, 1:].copy()

Xdf = Xdf.apply(pd.to_numeric, errors="coerce")
valid_mask = ~Xdf.isna().any(axis=1)

n_all = len(df_feat)
n_ok = int(valid_mask.sum())
n_drop = n_all - n_ok

print(f"[INFO] laogang feature rows: total={n_all}, kept={n_ok}, dropped(NaN)={n_drop}")

smiles_keep = smiles_feat.loc[valid_mask].reset_index(drop=True)
X_lg = Xdf.loc[valid_mask].to_numpy(dtype=np.float32)

if X_lg.shape[1] != X_tr_all.shape[1]:
    raise ValueError(
        f"老港特征维度与训练特征维度不一致："
        f"laogang={X_lg.shape[1]} vs train={X_tr_all.shape[1]}。"
        "请确认老港特征列与训练特征列完全一致。"
    )

print("[INFO] laogang X:", X_lg.shape)


# =========================================================
# 4) 读取老港 24 标签预测二值结果
# =========================================================
df_bin_raw = pd.read_excel(LAOGANG_PRED_BIN, sheet_name=SHEET_NAME)

# 如果预测文件包含所有原始行，则按 valid_mask 同步筛选；
# 如果预测文件已经是筛选后的 kept 行，则直接使用。
if len(df_bin_raw) == n_all:
    df_bin = df_bin_raw.loc[valid_mask].reset_index(drop=True)
elif len(df_bin_raw) == n_ok:
    df_bin = df_bin_raw.reset_index(drop=True)
else:
    raise ValueError(
        f"预测二值文件行数无法与特征文件对齐："
        f"pred_rows={len(df_bin_raw)}, feature_total={n_all}, feature_kept={n_ok}。"
    )

missing_labels = [c for c in MALODOR_LABELS_24 if c not in df_bin.columns]
if missing_labels:
    raise ValueError(
        "老港二值预测文件缺少以下24标签列：\n"
        + "\n".join(missing_labels)
    )

Ybin = (
    df_bin[MALODOR_LABELS_24]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
    .to_numpy()
)

if Ybin.shape[1] != 24:
    raise ValueError(f"Ybin 应为24列，但当前为 {Ybin.shape[1]} 列。")

if X_lg.shape[0] != Ybin.shape[0]:
    raise ValueError(
        f"X_lg 与 Ybin 行数不一致：X_lg={X_lg.shape[0]} vs Ybin={Ybin.shape[0]}"
    )

print("[INFO] laogang Ybin:", Ybin.shape)


# =========================================================
# 5) 计算 rho / IA / AD 内外
# =========================================================
Ntr = X_tr_all.shape[0]

k_swd_eff = min(max(int(0.1 * Ntr), 1), K_SWD_CAP)
k_eff = max(int(float(K_RATIO) * Ntr), 1)
k_eff = min(k_eff, Ntr)

print(f"[INFO] AD params: a={A_WEIGHT}, eps={EPSILON}, k_ratio={K_RATIO}")
print(f"[INFO] Thresholds: rho_th={RHO_TH}, IA_th={IA_TH}")
print(f"[INFO] Ntr={Ntr}, k_eff={k_eff}, k_swd_eff={k_swd_eff}")

SWD, Xtr_n = compute_SWD_topk(
    X_tr_all,
    y_tr_all,
    k_swd_eff,
    A_WEIGHT,
    EPSILON
)

rho_lg, IA_lg = compute_rho_IA_given_SWD(
    Xtr_n,
    SWD,
    X_lg,
    k_eff,
    A_WEIGHT,
    EPSILON
)

in_domain = (rho_lg >= RHO_TH) & (IA_lg <= IA_TH)
out_domain = ~in_domain


# =========================================================
# 6) 计算恶臭命中
#    24标签全为0：未命中恶臭
#    24标签任意一个为1：命中恶臭
# =========================================================
hit_malodor = (Ybin.sum(axis=1) > 0)

n_total = len(hit_malodor)
n_hit_total = int(hit_malodor.sum())
n_nohit_total = n_total - n_hit_total

n_in = int(in_domain.sum())
n_out = int(out_domain.sum())

n_hit_in = int(hit_malodor[in_domain].sum()) if n_in > 0 else 0
n_hit_out = int(hit_malodor[out_domain].sum()) if n_out > 0 else 0

n_nohit_in = n_in - n_hit_in
n_nohit_out = n_out - n_hit_out

rate_total = n_hit_total / n_total * 100 if n_total > 0 else np.nan
rate_in = n_hit_in / n_in * 100 if n_in > 0 else np.nan
rate_out = n_hit_out / n_out * 100 if n_out > 0 else np.nan

cov_in = n_in / n_total * 100 if n_total > 0 else np.nan
cov_out = n_out / n_total * 100 if n_total > 0 else np.nan

print("\n========== 老港数据集 AD 内外恶臭命中统计 ==========")
print(f"Total samples : {n_total}")
print(f"In-AD samples : {n_in} / {n_total} = {cov_in:.2f}%")
print(f"Out-AD samples: {n_out} / {n_total} = {cov_out:.2f}%")

print("\n---------- 恶臭命中情况：24标签任意命中 ----------")
print(f"Overall hit : {n_hit_total} / {n_total} = {rate_total:.2f}%")
print(f"In-AD hit   : {n_hit_in} / {n_in} = {rate_in:.2f}%" if n_in > 0 else "In-AD hit   : NA")
print(f"Out-AD hit  : {n_hit_out} / {n_out} = {rate_out:.2f}%" if n_out > 0 else "Out-AD hit  : NA")

print("\n---------- 未命中情况：24标签全为0 ----------")
print(f"Overall no-hit : {n_nohit_total} / {n_total} = {100 - rate_total:.2f}%")
print(f"In-AD no-hit   : {n_nohit_in} / {n_in} = {100 - rate_in:.2f}%" if n_in > 0 else "In-AD no-hit   : NA")
print(f"Out-AD no-hit  : {n_nohit_out} / {n_out} = {100 - rate_out:.2f}%" if n_out > 0 else "Out-AD no-hit  : NA")


# =========================================================
# 7) 导出结果
# =========================================================
out_df = pd.DataFrame()
out_df["SMILES"] = smiles_keep

for c in MALODOR_LABELS_24:
    out_df[c] = df_bin[c].astype(int).values

out_df["label_sum_24"] = Ybin.sum(axis=1)
out_df["hit_malodor"] = hit_malodor.astype(int)

out_df["rho"] = rho_lg
out_df["IA"] = IA_lg
out_df["in_domain"] = in_domain.astype(int)

summary_df = pd.DataFrame([
    {
        "group": "Overall",
        "n_samples": n_total,
        "n_hit_malodor": n_hit_total,
        "n_no_hit": n_nohit_total,
        "hit_rate_pct": rate_total,
        "coverage_pct": 100.0,
    },
    {
        "group": "In-AD",
        "n_samples": n_in,
        "n_hit_malodor": n_hit_in,
        "n_no_hit": n_nohit_in,
        "hit_rate_pct": rate_in,
        "coverage_pct": cov_in,
    },
    {
        "group": "Out-AD",
        "n_samples": n_out,
        "n_hit_malodor": n_hit_out,
        "n_no_hit": n_nohit_out,
        "hit_rate_pct": rate_out,
        "coverage_pct": cov_out,
    },
])

with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="summary", index=False)
    out_df.to_excel(writer, sheet_name="details", index=False)

print(f"\n[SAVED] {OUT_PATH}")

[INFO] AD reference train X: (15024, 2595)
[INFO] AD reference train y: (15024, 24)
[INFO] laogang feature rows: total=143, kept=143, dropped(NaN)=0
[INFO] laogang X: (143, 2595)
[INFO] laogang Ybin: (143, 24)
[INFO] AD params: a=100.0, eps=1e-06, k_ratio=0.05
[INFO] Thresholds: rho_th=8.445e-08, IA_th=0.4438
[INFO] Ntr=15024, k_eff=751, k_swd_eff=300

========== 老港数据集 AD 内外恶臭命中统计 ==========
Total samples : 143
In-AD samples : 43 / 143 = 30.07%
Out-AD samples: 100 / 143 = 69.93%

---------- 恶臭命中情况：24标签任意命中 ----------
Overall hit : 106 / 143 = 74.13%
In-AD hit   : 40 / 43 = 93.02%
Out-AD hit  : 66 / 100 = 66.00%

---------- 未命中情况：24标签全为0 ----------
Overall no-hit : 37 / 143 = 25.87%
In-AD no-hit   : 3 / 43 = 6.98%
Out-AD no-hit  : 34 / 100 = 34.00%

[SAVED] ./laogang_AD_malodor_hit_summary.xlsx
